# 1. Environment

In [1]:
from pathlib import Path
import os
import re
import shutil
import pandas as pd
from pandas.errors import EmptyDataError

In [2]:
os.chdir("C:\\Users\\yatsyk\\Documents\\NextCloud_Synchro\\passim_BoH\\")

In [3]:
project_root =  Path(".")
source_dir = project_root / "04.CoMMa" / "00.data" / "02.csvs_pages"
metadata_path = project_root / "04.CoMMa" / "02.metadata" / "196_arkindex_Comma_URL_filled.csv"
concordance_dir = project_root / "01.data" / "05.metadata_from_bases" / "arkindex_pages_id_page_ordering"

project_root, source_dir, metadata_path, concordance_dir

(WindowsPath('.'),
 WindowsPath('04.CoMMa/00.data/02.csvs_pages'),
 WindowsPath('04.CoMMa/02.metadata/196_arkindex_Comma_URL_filled.csv'),
 WindowsPath('01.data/05.metadata_from_bases/arkindex_pages_id_page_ordering'))

# 2. Data fetching

## 2.1. List the csv to align

In [4]:
source_files = sorted(source_dir.glob("*.csv"))
file_list = pd.DataFrame(
    {
        "csv_filename": [path.name for path in source_files],
        "entity_id": [path.stem for path in source_files],
    }
)
print(f"Found {len(file_list)} files")
file_list

Found 196 files


,csv_filename,entity_id
0,Q106117.csv,Q106117
1,Q106514.csv,Q106514
2,Q106532.csv,Q106532
3,Q106533.csv,Q106533
4,Q106537.csv,Q106537
...,...,...
191,Q70010.csv,Q70010
192,Q70015.csv,Q70015
193,Q70040.csv,Q70040
194,Q77199.csv,Q77199


## 2.2. Get correspondance ms `entity_id` -> ms `ID Arkindex`

In [5]:
metadata_df = pd.read_csv(metadata_path, dtype=str).fillna("")
metadata_df["entity_id"] = metadata_df["bbma_data_entity"].str.rsplit("/", n=1).str[-1]
metadata_df = metadata_df.rename(columns={"ID Arkindex": "arkindex_id"})
metadata_df

,Manuscript H-ID,bbma_data_entity,arkindex_id,URL CoMMA,Structural annotation,partially annotated,arkindex_comma_pages_matched,Leroquais_annotation,entity_id
0,1356,https://data.biblissima.fr/entity/Q241713,6150a30a-bff0-4672-96c6-9c19896cdc5e,https://comma.inria.fr/doc/https%253A%252F%252...,False,False,True,False,Q241713
1,1429,https://data.biblissima.fr/entity/Q235097,77fcbfe1-faf8-4316-8f32-0155ed3a60c5,https://comma.inria.fr/doc/https%253A%252F%252...,False,True,True,False,Q235097
2,1430,https://data.biblissima.fr/entity/Q235098,034c35ae-17e3-4e51-9821-8cf1662b1253,https://comma.inria.fr/doc/https%253A%252F%252...,False,True,True,False,Q235098
3,1642,https://data.biblissima.fr/entity/Q235484,73e26638-6275-41b9-9502-39e27f01d93f,https://comma.inria.fr/doc/https%253A%252F%252...,False,False,True,False,Q235484
4,1829,https://data.biblissima.fr/entity/Q203702,b3b9759b-56e1-460d-b766-8cee25d6aaa5,https://comma.inria.fr/doc/https%253A%252F%252...,False,True,True,False,Q203702
...,...,...,...,...,...,...,...,...,...
191,3118,https://data.biblissima.fr/entity/Q209026,5fe2eace-2177-4fa0-8c84-2b94b3d88c00,https://comma.inria.fr/doc/https%253A%252F%252...,False,True,True,False,Q209026
192,3119,https://data.biblissima.fr/entity/Q209044,a9a01b20-b6cc-43e1-b3cd-4b57bb3fc270,https://comma.inria.fr/doc/https%253A%252F%252...,False,True,True,False,Q209044
193,3355,https://data.biblissima.fr/entity/Q209755,c0b8fea9-df2b-4518-a55e-5337fa54ce13,https://comma.inria.fr/doc/https%253A%252F%252...,False,True,True,False,Q209755
194,4496,https://data.biblissima.fr/entity/Q211784,bb70bf8b-f365-47a2-9795-37d7c788518c,https://comma.inria.fr/doc/https%253A%252F%252...,False,True,False,False,Q211784


In [6]:
matches = file_list.merge(
    metadata_df[["entity_id", "arkindex_id", "bbma_data_entity", "Manuscript H-ID"]],
    on="entity_id",
    how="left",
)

missing_arkindex = matches[matches["arkindex_id"].isna() | (matches["arkindex_id"] == "")]
if not missing_arkindex.empty:
    raise ValueError(
        "Found no ID Arkindex for : "
        + ", ".join(missing_arkindex["csv_filename"].tolist())
    )

matches

,csv_filename,entity_id,arkindex_id,bbma_data_entity,Manuscript H-ID
0,Q106117.csv,Q106117,0068e751-c249-4d8b-82f6-07933fe0cecd,https://data.biblissima.fr/entity/Q106117,2513
1,Q106514.csv,Q106514,70aadb9e-fc7f-4583-afef-de71ecca65c6,https://data.biblissima.fr/entity/Q106514,2653
2,Q106532.csv,Q106532,f9f1fca3-072f-4056-9144-8aae1d4a6504,https://data.biblissima.fr/entity/Q106532,2470
3,Q106533.csv,Q106533,e741feb8-6ac4-48f9-8dca-af231bb7b862,https://data.biblissima.fr/entity/Q106533,2452
4,Q106537.csv,Q106537,cbb23979-72fc-48a3-bf59-0963fc91c1ed,https://data.biblissima.fr/entity/Q106537,2426
...,...,...,...,...,...
191,Q70010.csv,Q70010,009c37f0-618a-4d58-b81a-511a36fa0fac,https://data.biblissima.fr/entity/Q70010,2648
192,Q70015.csv,Q70015,b8c8130e-6426-4500-87b2-0059ea8fc7f2,https://data.biblissima.fr/entity/Q70015,2649
193,Q70040.csv,Q70040,c2ec140e-ec75-4ead-9777-544c502a02f9,https://data.biblissima.fr/entity/Q70040,2674
194,Q77199.csv,Q77199,79770f30-cbc9-4bfa-8a6b-0dec987d1ad8,https://data.biblissima.fr/entity/Q77199,2754


## 2.3. For every csv find the matching concordance file for alignment

In [7]:
matches["concordance_path"] = matches["arkindex_id"].map(lambda ark_id: concordance_dir / f"{ark_id}.csv")
matches["concordance_exists"] = matches["concordance_path"].map(Path.exists)

missing_concordances = matches[~matches["concordance_exists"]]
if not missing_concordances.empty:
    raise FileNotFoundError(
        "No concordances found for : "
        + ", ".join(missing_concordances["csv_filename"].tolist())
    )

matches[["csv_filename", "entity_id", "arkindex_id", "concordance_path"]]

,csv_filename,entity_id,arkindex_id,concordance_path
0,Q106117.csv,Q106117,0068e751-c249-4d8b-82f6-07933fe0cecd,01.data\05.metadata_from_bases\arkindex_pages_...
1,Q106514.csv,Q106514,70aadb9e-fc7f-4583-afef-de71ecca65c6,01.data\05.metadata_from_bases\arkindex_pages_...
2,Q106532.csv,Q106532,f9f1fca3-072f-4056-9144-8aae1d4a6504,01.data\05.metadata_from_bases\arkindex_pages_...
3,Q106533.csv,Q106533,e741feb8-6ac4-48f9-8dca-af231bb7b862,01.data\05.metadata_from_bases\arkindex_pages_...
4,Q106537.csv,Q106537,cbb23979-72fc-48a3-bf59-0963fc91c1ed,01.data\05.metadata_from_bases\arkindex_pages_...
...,...,...,...,...
191,Q70010.csv,Q70010,009c37f0-618a-4d58-b81a-511a36fa0fac,01.data\05.metadata_from_bases\arkindex_pages_...
192,Q70015.csv,Q70015,b8c8130e-6426-4500-87b2-0059ea8fc7f2,01.data\05.metadata_from_bases\arkindex_pages_...
193,Q70040.csv,Q70040,c2ec140e-ec75-4ead-9777-544c502a02f9,01.data\05.metadata_from_bases\arkindex_pages_...
194,Q77199.csv,Q77199,79770f30-cbc9-4bfa-8a6b-0dec987d1ad8,01.data\05.metadata_from_bases\arkindex_pages_...


In [32]:
missing_concordances.to_csv("04.CoMMa\\02.metadata\\concordances_arkindex\\missing_concordances.csv")

In [8]:
empty_concordances = []

for path in matches["concordance_path"].dropna().unique():
    path = Path(path)
    if path.exists() and path.stat().st_size == 0:
        empty_concordances.append(path)

empty_concordances


[]

# 3. Add arkindex_page_id to CoMMa transcriptions

In [9]:
csv_dir = Path("04.CoMMa/00.data/02.csvs_pages")

for _, match in matches.iterrows():
    csv_filename = match["csv_filename"]
    concordance_path = Path(match["concordance_path"])

    csv_path = csv_dir / csv_filename

    if not csv_path.exists():
        print(f"CSV not found: {csv_path}")
        continue

    if not concordance_path.exists():
        print(f"Concordance not found: {concordance_path}")
        continue

    df = pd.read_csv(csv_path)
    concordance = pd.read_csv(concordance_path)

    if "page_ordering" not in df.columns:
        print(f"No page_ordering in CSV: {csv_path}")
        continue

    if "page_ordering" not in concordance.columns:
        print(f"No page_ordering in concordance: {concordance_path}")
        continue

    if "arkindex_page_id" not in concordance.columns:
        print(f"No arkindex_page_id in concordance: {concordance_path}")
        continue

    df = df.drop(columns=["arkindex_page_id"], errors="ignore")

    df = df.merge(
        concordance[["page_ordering", "arkindex_page_id"]],
        on="page_ordering",
        how="left",
        validate="one_to_one",
    )

    df.to_csv(csv_path, index=False)

    print(f"Updated: {csv_path}")

Updated: 04.CoMMa\00.data\02.csvs_pages\Q106117.csv
Updated: 04.CoMMa\00.data\02.csvs_pages\Q106514.csv
Updated: 04.CoMMa\00.data\02.csvs_pages\Q106532.csv
Updated: 04.CoMMa\00.data\02.csvs_pages\Q106533.csv
Updated: 04.CoMMa\00.data\02.csvs_pages\Q106537.csv
Updated: 04.CoMMa\00.data\02.csvs_pages\Q106540.csv
Updated: 04.CoMMa\00.data\02.csvs_pages\Q106541.csv
Updated: 04.CoMMa\00.data\02.csvs_pages\Q106543.csv
Updated: 04.CoMMa\00.data\02.csvs_pages\Q106545.csv
Updated: 04.CoMMa\00.data\02.csvs_pages\Q106616.csv
Updated: 04.CoMMa\00.data\02.csvs_pages\Q106852.csv
Updated: 04.CoMMa\00.data\02.csvs_pages\Q107627.csv
Updated: 04.CoMMa\00.data\02.csvs_pages\Q107628.csv
Updated: 04.CoMMa\00.data\02.csvs_pages\Q107656.csv
Updated: 04.CoMMa\00.data\02.csvs_pages\Q107662.csv
Updated: 04.CoMMa\00.data\02.csvs_pages\Q109291.csv
Updated: 04.CoMMa\00.data\02.csvs_pages\Q109292.csv
Updated: 04.CoMMa\00.data\02.csvs_pages\Q109588.csv
Updated: 04.CoMMa\00.data\02.csvs_pages\Q109589.csv
Updated: 04.

# 4. Rename files using arkindex_ms_id

In [10]:
source_dir = Path("04.CoMMa/00.data/02.csvs_pages")
target_dir = Path("04.CoMMa/00.data/03.csvs_arkindex")

target_dir.mkdir(parents=True, exist_ok=True)

for _, row in matches.iterrows():
    old_name = row["csv_filename"]
    arkindex_id = row["arkindex_id"]

    source_path = source_dir / old_name
    target_path = target_dir / f"{arkindex_id}.csv"

    if not source_path.exists():
        print(f"Source file not found: {source_path}")
        continue

    shutil.copy2(source_path, target_path)

    print(f"Copied: {source_path.name} -> {target_path.name}")


Copied: Q106117.csv -> 0068e751-c249-4d8b-82f6-07933fe0cecd.csv
Copied: Q106514.csv -> 70aadb9e-fc7f-4583-afef-de71ecca65c6.csv
Copied: Q106532.csv -> f9f1fca3-072f-4056-9144-8aae1d4a6504.csv
Copied: Q106533.csv -> e741feb8-6ac4-48f9-8dca-af231bb7b862.csv
Copied: Q106537.csv -> cbb23979-72fc-48a3-bf59-0963fc91c1ed.csv
Copied: Q106540.csv -> ce56cfe8-390c-4227-95d2-211bb48b4e4b.csv
Copied: Q106541.csv -> 6a06c371-4b5d-4f33-b1d9-398b6c2b90c4.csv
Copied: Q106543.csv -> 739f2469-8006-4ed5-acdb-29e748c6af92.csv
Copied: Q106545.csv -> e9cc30a1-1528-411c-970c-57b0e417d964.csv
Copied: Q106616.csv -> 0253a5a1-0667-4c1c-b499-a59d48abc884.csv
Copied: Q106852.csv -> 26c036eb-c249-4720-9229-aa622ad736c2.csv
Copied: Q107627.csv -> ff5abf5d-527f-4db5-9ae0-51222fe71a78.csv
Copied: Q107628.csv -> f2fe9aa4-3856-4f09-ac5a-51b082a3b8e8.csv
Copied: Q107656.csv -> 17a645dc-d1e0-4bbb-8c48-f48ecf5bf4b9.csv
Copied: Q107662.csv -> edde1308-14da-45db-aa7b-a69826fc9324.csv
Copied: Q109291.csv -> f09f92d0-668e-4b3